In [1]:
from app import app
from app.extensions import db
from app.database.models import Amarnameh
from sqlalchemy import func
import pandas as pd

with app.app_context():
    
    data = pd.read_csv("app/assets/data/amarnameh.csv")
    
    for i, row in data.iterrows():
        ri = Amarnameh(
            ostan = row['ostan'],
            shahrestan = row['shahrestan'],
            bakhsh = row['bakhsh'],
            shahr = row['shahr'],
            region = int(row['region']),
            district = int(row['district']),
            area = int(row['area']),
            population = int(row['population']),
            population_male = int(row['population_male']),
            population_female = int(row['population_female']),
            n_households = int(row['n_households']),
        )
        db.session.add(ri)
    db.session.commit()


In [ ]:
from app import app
from app.extensions import db
from app.database.models import Bakery, OwnershipStatus, SecondFuel, HouseholdRisk, BakersRisk, TypeFlour, TypeBread
from sqlalchemy import func
import pandas as pd

originalCols = [
    'first_name',
    'last_name',
    'nid',
    'phone',
    'bakery_id',
    'ownership_status',
    'number_violations',
    'second_fuel',
    'city',
    'lat',
    'lon',
    'household_risk',
    'bakers_risk',
    'flour_types',
    'bread_types',
    'bread_rations'
]

with app.app_context():
        
    df = pd.read_csv("C:\\Users\\pooya\\Desktop\\upload.csv")
    
    # Check Columns Name
    if df.shape[1] != originalCols.__len__():
        
        

    


16
16


In [15]:
def validate_iranian_national_code(code):
    code_len = len(code)
    if code_len > 10 or code_len < 8:
        return False

    if len(set(code)) == 1:
        return False

    if len(code) < 10:
        code = code.zfill(10)

    factors = [10, 9, 8, 7, 6, 5, 4, 3, 2]
    checksum = sum(int(code[i]) * factors[i] for i in range(len(code) - 1))
    remainder = checksum % 11
    last_digit = int(code[-1])

    if remainder < 2:
        return remainder == last_digit
    else:
        return 11 - remainder == last_digit


validate_iranian_national_code("1063555566")

True

In [17]:
"7  886992  091".replace(' +', '')

'7  886992  091'

In [31]:
from app import app
from app.extensions import db
from app.database.models import Bakery, Amarnameh
from sqlalchemy import func
import pandas as pd
import json

with app.app_context():
    
    bakery_group  = db.session.query(
        Bakery.ostan,
        Bakery.shahrestan,
        Bakery.bakhsh,
        Bakery.shahr,
        Bakery.region,
        func.count(Bakery.id).label('bakery_count'),
        func.sum(Bakery.bread_rations).label('bread_rations'),
    ).group_by(
        Bakery.ostan,
        Bakery.shahrestan,
        Bakery.bakhsh,
        Bakery.shahr,
        Bakery.region
    ).subquery()
    
    result = db.session.query(
        bakery_group.c.ostan,
        bakery_group.c.shahrestan,
        bakery_group.c.bakhsh,
        bakery_group.c.shahr,
        bakery_group.c.region,
        bakery_group.c.bakery_count,
        bakery_group.c.bread_rations,
        func.sum(Amarnameh.area).label('area'),
        func.sum(Amarnameh.population).label('total_population'),
        func.sum(Amarnameh.population_male).label('male_population'),
        func.sum(Amarnameh.population_female).label('female_population'),
        func.sum(Amarnameh.n_households).label('total_households'),
        (func.sum(Amarnameh.population) / bakery_group.c.bakery_count).label('population_per_bakery'),
        (bakery_group.c.bread_rations * 100 / func.sum(Amarnameh.population)).label('ration_per_population_per_100')
    ).join(
        Amarnameh,
        (bakery_group.c.ostan == Amarnameh.ostan) &
        (bakery_group.c.shahrestan == Amarnameh.shahrestan) &
        (bakery_group.c.bakhsh == Amarnameh.bakhsh) &
        (bakery_group.c.shahr == Amarnameh.shahr) &
        (bakery_group.c.region == Amarnameh.region)
    ).group_by(
        bakery_group.c.ostan,
        bakery_group.c.shahrestan,
        bakery_group.c.bakhsh,
        bakery_group.c.shahr,
        bakery_group.c.region
    ).all()
    
    json_data = [
        {
            "ostan": row.ostan,
            "shahrestan": row.shahrestan,
            "bakhsh": row.bakhsh,
            "shahr": row.shahr,
            "region": row.region,
            "bakery_count": row.bakery_count,
            "bread_rations": row.bread_rations,
            "area": row.area,
            "total_population": row.total_population,
            "male_population": row.male_population,
            "female_population": row.female_population,
            "total_households": row.total_households
        }
        for row in result
    ]

# Convert the list to a JSON string
json_response = json.dumps(json_data, ensure_ascii=False)

# Print or return the JSON response
print(json_response)
        
    


[{"ostan": "خراسان رضوی", "shahrestan": "مشهد", "bakhsh": "احمدآباد", "shahr": "بینالود", "region": 1, "bakery_count": 3, "bread_rations": 553, "area": 838.0, "total_population": 5635, "male_population": 2815, "female_population": 2820, "total_households": 1770}, {"ostan": "خراسان رضوی", "shahrestan": "مشهد", "bakhsh": "احمدآباد", "shahr": "روستایی", "region": 1, "bakery_count": 35, "bread_rations": 5490, "area": 0.0, "total_population": 30932, "male_population": 15712, "female_population": 15220, "total_households": 9571}, {"ostan": "خراسان رضوی", "shahrestan": "مشهد", "bakhsh": "احمدآباد", "shahr": "ملک آباد", "region": 1, "bakery_count": 3, "bread_rations": 671, "area": 101.0, "total_population": 2056, "male_population": 1053, "female_population": 1003, "total_households": 620}, {"ostan": "خراسان رضوی", "shahrestan": "مشهد", "bakhsh": "رضویه", "shahr": "رضویه", "region": 1, "bakery_count": 2, "bread_rations": 244, "area": 142.0, "total_population": 8850, "male_population": 4316, "fe

In [3]:
from app import app
from app.extensions import db
from app.database.models import Bakery, Amarnameh
from sqlalchemy import func
import pandas as pd
import json

with app.app_context():
    query = db.session.query(Bakery)
    df = pd.read_sql(str(query.statement), db.engine)
    df.to_csv("a.csv", index=False, encoding='utf-8')
    